## 4. Mã nguồn cho Notebook 2: `notebooks/02_data_processing.ipynb` (Chia dữ liệu chống Data Leakage & Tiền xử lý)

Quy trình dưới đây tuân thủ nghiêm ngặt nguyên tắc **chống rò rỉ dữ liệu (Zero Data Leakage)**:
1. Thực hiện chia tách `Train` / `Test` có phân tầng (`stratify`) **ngay từ đầu** trước khi tính toán bất kỳ tham số thống kê nào.
2. Mọi tham số tiền xử lý (`valid_age_median`, `valid_reg_median`, danh sách cột `log1p_cols`, và các tham số `median`, `IQR`/`std` của `RobustScaler`) đều chỉ được `fit` (tính toán) trên tập **Train**, sau đó áp dụng (`transform`) sang tập **Test**.

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

RAW_DATA_PATH = "../data/raw/dataset_B_05_2020.csv"
PROCESSED_DIR = "../data/processed"
os.makedirs(PROCESSED_DIR, exist_ok=True)

# 1. Đọc dữ liệu gốc
df = pd.read_csv(RAW_DATA_PATH)

# 2. Mã hóa nhãn mục tiêu: legitimate -> 0, phishing -> 1
df['status'] = (df['status'] == 'phishing').astype(int)

# 3. Loại bỏ cột định danh 'url' và 6 cột hằng số (Zero-variance)
zero_var_cols = [
    'nb_or', 'ratio_nullHyperlinks', 'ratio_intRedirection',
    'ratio_intErrors', 'submit_email', 'sfh'
]
df_clean = df.drop(columns=['url'] + zero_var_cols)

# 4. CHIA DỮ LIỆU TRAIN / TEST (80/20, STRATIFIED) TRƯỚC KHI TIỀN XỬ LÝ
train_df, test_df = train_test_split(
    df_clean,
    test_size=0.20,
     random_state=42,
    stratify=df_clean['status']
)
train_df = train_df.copy().reset_index(drop=True)
test_df = test_df.copy().reset_index(drop=True)

print(f"Kích thước tập Train: {train_df.shape} (Tỷ lệ Phishing: {train_df['status'].mean():.2%})")
print(f"Kích thước tập Test:  {test_df.shape} (Tỷ lệ Phishing: {test_df['status'].mean():.2%})")

# 5. Xử lý giá trị khuyết ẩn (Sentinel values < 0) - CHỈ TÍNH MEDIAN TRÊN TẬP TRAIN
# 5a. Cột domain_age (< 0 nghĩa là không tra cứu được WHOIS)
train_df['domain_age_unknown'] = (train_df['domain_age'] < 0).astype(int)
test_df['domain_age_unknown'] = (test_df['domain_age'] < 0).astype(int)

valid_age_median = float(train_df.loc[train_df['domain_age'] >= 0, 'domain_age'].median())
train_df['domain_age'] = train_df['domain_age'].astype(float)
test_df['domain_age'] = test_df['domain_age'].astype(float)
train_df.loc[train_df['domain_age'] < 0, 'domain_age'] = valid_age_median
test_df.loc[test_df['domain_age'] < 0, 'domain_age'] = valid_age_median

# 5b. Cột domain_registration_length (<= 0 nghĩa là khuyết thời hạn đăng ký)
train_df['domain_reg_len_unknown'] = (train_df['domain_registration_length'] <= 0).astype(int)
test_df['domain_reg_len_unknown'] = (test_df['domain_registration_length'] <= 0).astype(int)

valid_reg_median = float(train_df.loc[train_df['domain_registration_length'] > 0, 'domain_registration_length'].median())
train_df['domain_registration_length'] = train_df['domain_registration_length'].astype(float)
test_df['domain_registration_length'] = test_df['domain_registration_length'].astype(float)
train_df.loc[train_df['domain_registration_length'] <= 0, 'domain_registration_length'] = valid_reg_median
test_df.loc[test_df['domain_registration_length'] <= 0, 'domain_registration_length'] = valid_reg_median

# 6. Đồng nhất các cột phần trăm [0, 100] về tỷ lệ [0, 1]
pct_cols = ['links_in_tags', 'ratio_intMedia', 'ratio_extMedia', 'safe_anchor']
for c in pct_cols:
    train_df[c] = train_df[c] / 100.0
    test_df[c] = test_df[c] / 100.0

# 7. Phân tách cột Nhị phân (0/1) và cột Liên tục/Đếm trên tập Train
feature_cols = [c for c in train_df.columns if c != 'status']
binary_cols = [c for c in feature_cols if set(train_df[c].unique()).issubset({0, 1})]
cont_cols = [c for c in feature_cols if c not in binary_cols]

# 8. Biến đổi Log1p cho các cột liên tục/đếm bị lệch phải nặng (Skewness > 1.0 trên Train)
train_skew = train_df[cont_cols].skew()
log1p_cols = train_skew[train_skew > 1.0].index.tolist()

for c in log1p_cols:
    train_df[c] = np.log1p(np.maximum(0, train_df[c]))
    test_df[c] = np.log1p(np.maximum(0, test_df[c]))

# 9. Chuẩn hóa thang đo (Robust Scaling kết hợp Fallback Std cho cột thưa) - FIT TRÊN TRAIN
for c in cont_cols:
    med = train_df[c].median()
    q25 = train_df[c].quantile(0.25)
    q75 = train_df[c].quantile(0.75)
    iqr = q75 - q25
    # Nếu cột thưa có IQR == 0, sử dụng độ lệch chuẩn (std) của Train để tránh chia cho 0
    scale = iqr if iqr > 1e-6 else train_df[c].std()
    if scale > 1e-9:
        train_df[c] = (train_df[c] - med) / scale
        test_df[c] = (test_df[c] - med) / scale

# 10. Sắp xếp cột nhãn 'status' về cuối và xuất file CSV
final_cols = [c for c in train_df.columns if c != 'status'] + ['status']
train_df = train_df[final_cols]
test_df = test_df[final_cols]

train_df.to_csv(f"{PROCESSED_DIR}/train.csv", index=False)
test_df.to_csv(f"{PROCESSED_DIR}/test.csv", index=False)
print("Đã lưu thành công ../data/processed/train.csv và ../data/processed/test.csv!")